In [1]:
from pathlib import Path
import json
import random

In [2]:
FULL_JSON = Path("../dataset/full.json")
OURSRC_JSON = Path("../dataset/oursrc.json")
OUTPUT_DIR = Path("../dataset")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42
REMOVE_DUPLICATES = True

In [3]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON array")
    return data


def record_signature(item: dict):
    return (
        (item.get("instruction") or "").strip(),
        (item.get("answer") or "").strip(),
        (item.get("problem") or "").strip(),
    )


def deduplicate(records):
    seen = set()
    unique = []
    for item in records:
        sig = record_signature(item)
        if sig in seen:
            continue
        seen.add(sig)
        unique.append(item)
    return unique


def split_records(records, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 1e-9:
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    rng = random.Random(seed)
    shuffled = records[:]
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    # Keep all remaining items in test so no sample is lost.
    n_test = n - n_train - n_val

    train = shuffled[:n_train]
    val = shuffled[n_train : n_train + n_val]
    test = shuffled[n_train + n_val : n_train + n_val + n_test]

    return train, val, test


def save_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [5]:
# 1) Load
full_data = load_json(FULL_JSON)
oursrc_data = load_json(OURSRC_JSON)

# 2) Merge
merged = [dict(item) for item in full_data] + [dict(item) for item in oursrc_data]

# 3) Optional dedup
if REMOVE_DUPLICATES:
    merged = deduplicate(merged)

# 4) Split
train_data, val_data, test_data = split_records(
    merged,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)

# 5) Save
train_path = OUTPUT_DIR / "train.json"
val_path = OUTPUT_DIR / "val.json"
test_path = OUTPUT_DIR / "test.json"

save_json(train_path, train_data)
save_json(val_path, val_data)
save_json(test_path, test_data)

print(f"Loaded full.json: {len(full_data)}")
print(f"Loaded oursrc.json: {len(oursrc_data)}")
print(f"Merged records: {len(merged)}")
print(f"Train: {len(train_data)} -> {train_path}")
print(f"Val:   {len(val_data)} -> {val_path}")
print(f"Test:  {len(test_data)} -> {test_path}")

Loaded full.json: 898
Loaded oursrc.json: 183
Merged records: 962
Train: 769 -> ..\dataset\train.json
Val:   96 -> ..\dataset\val.json
Test:  97 -> ..\dataset\test.json
